# Matrix factorization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Learning low-dimensional user and item factors
Matrix factorization explains ratings as dot products between user and item vectors. These five examples show reconstruction, one SGD update, regularization, rank as a capacity knob, and prediction for a missing entry.

In [ ]:
# 1) a rank-2 dot product reconstructs known ratings approximately
P=np.array([[1.2,0.2],[1.1,0.3],[0.1,1.3],[0.2,1.2]])
Q=np.array([[4.0,0.2],[3.8,0.3],[0.2,3.7],[0.4,3.5]])
Rhat=P@Q.T
plt.figure(figsize=(5,4)); plt.imshow(Rhat,cmap='viridis'); plt.colorbar(label='predicted rating'); plt.title('P @ Q.T')
assert abs(Rhat[0,0]-4.84)<1e-9 and abs(Rhat[2,2]-4.83)<1e-9

In [ ]:
# 2) one SGD update moves a low prediction upward
pu=np.array([0.8,0.2]); qi=np.array([3.0,0.4]); r=5.0; eta=0.05; lam=0.1
pred=float(pu@qi); err=r-pred
pu2=pu+eta*(err*qi-lam*pu); qi2=qi+eta*(err*pu-lam*qi); pred2=float(pu2@qi2)
plt.figure(figsize=(6,3)); plt.bar(['before','after'],[pred,pred2]); plt.title(f'{pred:.3f} -> {pred2:.3f}')
assert abs(pred-2.48)<1e-9 and pred2>pred

In [ ]:
# 3) L2 regularization shrinks factor norms during the same update
pu=np.array([0.8,0.2]); qi=np.array([3.0,0.4]); r=5.0; eta=0.05
vals=[]
for lam in [0,0.1,1.0]:
    err=r-float(pu@qi); p2=pu+eta*(err*qi-lam*pu); q2=qi+eta*(err*pu-lam*qi); vals.append(np.linalg.norm(p2)+np.linalg.norm(q2))
plt.figure(figsize=(6,3)); plt.bar(['0','.1','1.0'],vals); plt.xlabel('lambda'); plt.ylabel('sum norms'); plt.title('larger lambda shrinks updates')
assert vals[2]<vals[0]

In [ ]:
# 4) rank controls capacity: SVD reconstruction error drops with rank
R=np.array([[5,4,1,1],[4,5,1,1],[1,1,5,4],[1,1,4,5]],float)
U,S,Vt=np.linalg.svd(R,full_matrices=False); errs=[]
for k in [1,2,3,4]: errs.append(np.linalg.norm(R-(U[:,:k]*S[:k])@Vt[:k], 'fro'))
plt.figure(figsize=(6,3)); plt.plot([1,2,3,4],errs,'-o'); plt.xlabel('rank k'); plt.ylabel('Frobenius error'); plt.title('higher rank fits more')
assert abs(errs[1]-1.4142135623730951)<1e-9 and errs[-1]<1e-12

In [ ]:
# 5) filling a missing entry by factor dot product
P=np.array([[1.2,0.2],[1.1,0.3],[0.1,1.3],[0.2,1.2]])
Q=np.array([[4.0,0.2],[3.8,0.3],[0.2,3.7],[0.4,3.5]])
pred=float(P[0]@Q[2])
plt.figure(figsize=(6,3)); plt.quiver([0,0],[0,0],[P[0,0],Q[2,0]],[P[0,1],Q[2,1]],angles='xy',scale_units='xy',scale=1); plt.xlim(0,1.5); plt.ylim(0,4); plt.title(f'missing rating = dot = {pred:.3f}')
assert abs(pred-0.98)<1e-9